# BÁO CÁO THỰC HÀNH & DEMO - THÀNH VIÊN 3
## ĐỀ TÀI: DEEP GRAPH LEARNING IN AI - DỰ ĐOÁN BỆNH TIỂU ĐƯỜNG (DIABETES PREDICTION)
**Môn học:** Phát triển Hệ thống Thông minh  
**Thành viên thực hiện:** Thành viên 3 (Nghiên cứu bài toán y sinh, Tiền xử lý dữ liệu, Khởi tạo đồ thị bệnh nhân & Phòng chống Data Leakage)  
**Tập dữ liệu:** Diabetes Clinical Dataset (100,000 bản ghi lâm sàng, 9 đặc trưng)  
**Bàn giao cho:** Thành viên 1 & 4 (Mô hình hóa Graph Neural Network - GCN / GraphSAGE / GAT)


In [1]:
# 1. KHỞI TẠO MÔI TRƯỜNG & THƯ VIỆN CẦN THIẾT
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.neighbors import NearestNeighbors
import torch
from torch_geometric.data import Data
import warnings
warnings.filterwarnings('ignore')

# Thiết lập hiển thị đồ thị đẹp
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['axes.edgecolor'] = '#cbd5e0'
plt.rcParams['axes.linewidth'] = 0.8
print("Thư viện đã được nạp thành công! PyTorch version:", torch.__version__)


Thư viện đã được nạp thành công! PyTorch version: 2.14.0+cpu


C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


---
### PHẦN 1: KHÁM PHÁ & ĐÁNH GIÁ TẬP DỮ LIỆU LÂM SÀNG (EDA)
Đọc dữ liệu từ `data/diabetes_100k.csv`, phân tích các cột đặc trưng, kiểm tra khuyết thiếu và mức độ mất cân bằng nhãn.


In [2]:
# Nạp dữ liệu 100k bệnh nhân
data_path = Path("data/diabetes_100k.csv")
df = pd.read_csv(data_path)

print(f"[*] Kích thước bảng dữ liệu gốc: {df.shape[0]:,} dòng x {df.shape[1]} cột")
print("[*] Kiểm tra giá trị khuyết thiếu (Missing values):", df.isnull().sum().to_dict())
df.head(5)


[*] Kích thước bảng dữ liệu gốc: 100,000 dòng x 9 cột
[*] Kiểm tra giá trị khuyết thiếu (Missing values): {'gender': 0, 'age': 0, 'hypertension': 0, 'heart_disease': 0, 'smoking_history': 0, 'bmi': 0, 'HbA1c_level': 0, 'blood_glucose_level': 0, 'diabetes': 0}


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [3]:
# Thống kê phân bố nhãn mục tiêu và chuẩn vàng ADA
counts = df['diabetes'].value_counts()
print("Phân bố nhãn mục tiêu (diabetes):")
print(f" - Lớp 0 (Âm tính): {counts[0]:,} ({counts[0]/len(df)*100:.2f}%)")
print(f" - Lớp 1 (Dương tính): {counts[1]:,} ({counts[1]/len(df)*100:.2f}%)")
print(f" -> Tỷ lệ mất cân bằng (Imbalance Ratio): {counts[0]/counts[1]:.2f} : 1")

print("\nĐối chiếu với tiêu chuẩn ADA (American Diabetes Association):")
print(f" - HbA1c trung bình: Nhóm Bệnh = {df[df['diabetes']==1]['HbA1c_level'].mean():.2f}% (≥ 6.5%) vs Nhóm Khỏe = {df[df['diabetes']==0]['HbA1c_level'].mean():.2f}%")
print(f" - Glucose trung bình: Nhóm Bệnh = {df[df['diabetes']==1]['blood_glucose_level'].mean():.1f} mg/dL vs Nhóm Khỏe = {df[df['diabetes']==0]['blood_glucose_level'].mean():.1f} mg/dL")


Phân bố nhãn mục tiêu (diabetes):
 - Lớp 0 (Âm tính): 91,500 (91.50%)
 - Lớp 1 (Dương tính): 8,500 (8.50%)
 -> Tỷ lệ mất cân bằng (Imbalance Ratio): 10.76 : 1

Đối chiếu với tiêu chuẩn ADA (American Diabetes Association):
 - HbA1c trung bình: Nhóm Bệnh = 6.93% (≥ 6.5%) vs Nhóm Khỏe = 5.40%
 - Glucose trung bình: Nhóm Bệnh = 194.1 mg/dL vs Nhóm Khỏe = 132.9 mg/dL


---
### PHẦN 2: TIỀN XỬ LÝ & THIẾT LẬP PHÒNG THỦ CHỐNG DATA LEAKAGE
1. **Chia Stratified Split (80/20) TRƯỚC TIÊN**: Bảo toàn tỷ lệ mất cân bằng trên Train và Test.
2. **Loại bỏ cột `diabetes` ra khỏi không gian tính toán**: Tuyệt đối không dùng nhãn để tìm láng giềng KNN (Chống Label Leakage).
3. **Sử dụng `RobustScaler` (Median & IQR)**: Fit DUY NHẤT trên Train, Transform sang Test (Chống Statistical Leakage & miễn nhiễm với ngoại lệ y sinh).


In [4]:
# Lấy mẫu đại diện phân tầng 10,000 bệnh nhân để xử lý và tạo đồ thị tối ưu
SAMPLE_SIZE = 10000
df_sample = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

X = df_sample.drop(columns=['diabetes'])
y = df_sample['diabetes'].values

# Stratified Split 80/20
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

cat_cols = ['gender', 'smoking_history']
num_cols = ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level']

# Tiền xử lý số: RobustScaler
scaler = RobustScaler()
X_train_num = scaler.fit_transform(X_train_raw[num_cols])
X_test_num = scaler.transform(X_test_raw[num_cols])

# Tiền xử lý hạng mục: OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_cat = encoder.fit_transform(X_train_raw[cat_cols])
X_test_cat = encoder.transform(X_test_raw[cat_cols])

# Ghép đặc trưng
X_train = np.hstack([X_train_num, X_train_cat])
X_test = np.hstack([X_test_num, X_test_cat])

print(f"[*] Kích thước tập Train: {X_train.shape[0]:,} mẫu | Số chiều đặc trưng d = {X_train.shape[1]}")
print(f"[*] Kích thước tập Test: {X_test.shape[0]:,} mẫu | Số chiều đặc trưng d = {X_test.shape[1]}")
print("[+] Xác nhận: Không có biến 'diabetes' trong ma trận X. Scaler fit 100% độc lập trên Train.")


[*] Kích thước tập Train: 8,000 mẫu | Số chiều đặc trưng d = 14
[*] Kích thước tập Test: 2,000 mẫu | Số chiều đặc trưng d = 14
[+] Xác nhận: Không có biến 'diabetes' trong ma trận X. Scaler fit 100% độc lập trên Train.


---
### PHẦN 3: XÂY DỰNG ĐỒ THỊ TƯƠNG ĐỒNG BỆNH NHÂN (02 PHƯƠNG ÁN)
- **Phương án 1 (Baseline Homogeneous KNN)**: $k=10$, khoảng cách Euclidean đồng nhất.
- **Phương án 2 (Clinically-Weighted KNN - Đề xuất)**: $k=8$, nhân trọng số ưu tiên y sinh ($w_{\text{HbA1c}}=3, w_{\text{Glucose}}=3, w_{\text{BMI}}=2$).
- **Trọng số cạnh**: Sử dụng Gaussian RBF Kernel $W_{ij} = \exp\left(-\frac{d_{ij}^2}{2\sigma^2}\right)$.


In [5]:
# 1. PHƯƠNG ÁN 1: Baseline KNN (k=10)
nbrs1 = NearestNeighbors(n_neighbors=11, metric='euclidean', n_jobs=-1).fit(X_train)
distances1, indices1 = nbrs1.kneighbors(X_train)

same1, total1 = 0, 0
for i in range(len(X_train)):
    for rank in range(1, 11):
        j = indices1[i, rank]
        total1 += 1
        if y_train[i] == y_train[j]:
            same1 += 1
homophily_pa1 = same1 / total1

# 2. PHƯƠNG ÁN 2: Clinically-Weighted KNN (k=8)
feat_weights = np.ones(X_train.shape[1], dtype=np.float32)
feat_weights[num_cols.index('HbA1c_level')] = 3.0
feat_weights[num_cols.index('blood_glucose_level')] = 3.0
feat_weights[num_cols.index('bmi')] = 2.0

X_train_weighted = X_train * np.sqrt(feat_weights)
nbrs2 = NearestNeighbors(n_neighbors=9, metric='euclidean', n_jobs=-1).fit(X_train_weighted)
distances2, indices2 = nbrs2.kneighbors(X_train_weighted)

same2, total2 = 0, 0
src_nodes, dst_nodes, edge_weights = [], [], []
sigma2 = np.median(distances2[:, 1:])

for i in range(len(X_train)):
    for rank in range(1, 9):
        j = indices2[i, rank]
        dist = distances2[i, rank]
        w = np.exp(- (dist**2) / (2 * (sigma2**2) + 1e-8))
        src_nodes.append(i)
        dst_nodes.append(j)
        edge_weights.append(w)
        
        total2 += 1
        if y_train[i] == y_train[j]:
            same2 += 1
homophily_pa2 = same2 / total2

print(f"=== SO SÁNH HIỆU QUẢ 2 PHƯƠNG ÁN THIẾT KẾ ĐỒ THỊ ===")
print(f"PA1 (Baseline KNN k=10):            {total1:,} cạnh | Homophily Ratio h = {homophily_pa1:.2%}")
print(f"PA2 (Clinically-Weighted k=8):      {total2:,} cạnh | Homophily Ratio h = {homophily_pa2:.2%}")
print(f"-> KẾT LUẬN: PA2 tăng độ thuần cụm +{(homophily_pa2-homophily_pa1)*100:.2f}% và tiết kiệm 20% bộ nhớ cạnh!")


=== SO SÁNH HIỆU QUẢ 2 PHƯƠNG ÁN THIẾT KẾ ĐỒ THỊ ===
PA1 (Baseline KNN k=10):            80,000 cạnh | Homophily Ratio h = 93.36%
PA2 (Clinically-Weighted k=8):      64,000 cạnh | Homophily Ratio h = 94.11%
-> KẾT LUẬN: PA2 tăng độ thuần cụm +0.75% và tiết kiệm 20% bộ nhớ cạnh!


---
### PHẦN 4: THIẾT LẬP ĐỒ THỊ QUY NẠP (INDUCTIVE GRAPH SETTING)
Phòng chống **Transductive Leakage**: Nút Test chỉ truy vấn $k$ láng giềng trong Train để nhận thông điệp tổng hợp. Tuyệt đối **không** tạo cạnh giữa các nút Test (Test-Test = 0).


In [6]:
# Thiết lập kết nối Inductive cho Test
X_test_weighted = X_test * np.sqrt(feat_weights)
test_dist, test_idx = nbrs2.kneighbors(X_test_weighted, n_neighbors=8)

test_src, test_dst = [], []
for t_idx in range(len(X_test)):
    global_t_idx = len(X_train) + t_idx
    for rank in range(8):
        tr_neighbor = test_idx[t_idx, rank]
        test_src.append(tr_neighbor)
        test_dst.append(global_t_idx)

print(f"[*] Số cạnh nối Inductive từ Train sang Test: {len(test_src):,} cạnh.")
print("[+] Xác nhận: Không có cạnh nội bộ Test-Test. Bảo đảm tính khách quan khi inference!")


[*] Số cạnh nối Inductive từ Train sang Test: 16,000 cạnh.
[+] Xác nhận: Không có cạnh nội bộ Test-Test. Bảo đảm tính khách quan khi inference!


---
### PHẦN 5: ĐÓNG GÓI CHUẨN PYTORCH GEOMETRIC (DATA OBJECT)
Đóng gói đồ thị thành đối tượng `torch_geometric.data.Data(x, edge_index, edge_attr, y)` và lưu file `.pt` để bàn giao cho Thành viên 1 & 4.


In [7]:
# Đóng gói đồ thị PyG
x_tensor = torch.tensor(X_train, dtype=torch.float32)
edge_index = torch.tensor([src_nodes, dst_nodes], dtype=torch.int64)
edge_attr = torch.tensor(edge_weights, dtype=torch.float32).unsqueeze(1)
y_tensor = torch.tensor(y_train, dtype=torch.int64)

graph_data = Data(
    x=x_tensor,
    edge_index=edge_index,
    edge_attr=edge_attr,
    y=y_tensor
)

print("=== ĐỐI TƯỢNG ĐỒ THỊ PYTORCH GEOMETRIC HOÀN CHỈNH ===")
print(graph_data)
print(f" - Số lượng nút: {graph_data.num_nodes:,}")
print(f" - Số lượng cạnh: {graph_data.num_edges:,}")
print(f" - Số chiều đặc trưng nút: {graph_data.num_node_features}")
print(f" - Kích thước edge_index: {list(graph_data.edge_index.shape)}")
print(f" - Kích thước edge_attr: {list(graph_data.edge_attr.shape)}")

# Lưu đĩa
out_path = Path("data/patient_graph_tv3_demo.pt")
out_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(graph_data, out_path)
print(f"\n[+] ĐÃ LƯU FILE BÀN GIAO THÀNH CÔNG: {out_path.resolve()}")


=== ĐỐI TƯỢNG ĐỒ THỊ PYTORCH GEOMETRIC HOÀN CHỈNH ===
Data(x=[8000, 14], edge_index=[2, 64000], edge_attr=[64000, 1], y=[8000])
 - Số lượng nút: 8,000
 - Số lượng cạnh: 64,000
 - Số chiều đặc trưng nút: 14
 - Kích thước edge_index: [2, 64000]
 - Kích thước edge_attr: [64000, 1]

[+] ĐÃ LƯU FILE BÀN GIAO THÀNH CÔNG: D:\learning\phat_trien_he_thong_thong_minh\data\patient_graph_tv3_demo.pt
